# End-to-End Machine Learning Workflow: Diabetes Prediction

This tutorial demonstrates a complete, professional machine learning workflow using the **TuiML** library. We will build a model to predict diabetes using the classic Pima Indians Diabetes Database.

The workflow follows these structured steps:
1. **Problem Identification**
2. **Data Collection and Exploration (EDA)**
3. **Data Preprocessing**
4. **Feature Engineering and Selection**
5. **Dataset Splitting**
6. **Model Training and Selection using Experiments**
7. **Comparative Analysis**
8. **Model Deployment (Serving)**
9. **Calling the Model API**
10. **Simplified Workflows (The One-Liner Approach)**

## 1. Problem Identification

**Objective:** Predict whether a patient is diabetic based on medical diagnostic measurements.

**Task Type:** Binary Classification (Diabetic: 1, Non-diabetic: 0).

**Dataset Features:**
- **Pregnancies**: Number of times pregnant
- **Glucose**: Plasma glucose concentration
- **BloodPressure**: Diastolic blood pressure (mm Hg)
- **SkinThickness**: Triceps skin fold thickness (mm)
- **Insulin**: 2-Hour serum insulin (mu U/ml)
- **BMI**: Body mass index (weight in kg/(height in m)^2)
- **DiabetesPedigreeFunction**: Diabetes pedigree function
- **Age**: Age (years)
- **Outcome**: Class variable (target)

## 2. Data Collection and Exploration

We use TuiML's built-in dataset loaders to pull the diabetes data.

In [1]:
from tuiml.datasets import load_diabetes
import pandas as pd

# Load dataset
dataset = load_diabetes()
X, y = dataset.X, dataset.y
feature_names = dataset.feature_names

# Convert to Pandas for easy EDA
df = dataset.to_pandas()

print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (768, 9)


,preg,plas,pres,skin,insu,mass,pedi,age,target
0,6.0,148.0,72.0,35.0,0.0,33.6,0.627,50.0,tested_positive
1,1.0,85.0,66.0,29.0,0.0,26.6,0.351,31.0,tested_negative
2,8.0,183.0,64.0,0.0,0.0,23.3,0.672,32.0,tested_positive
3,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0,tested_negative
4,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0,tested_positive


### Exploratory Data Analysis (EDA)
Let's look at the statistical summary and class distribution.

In [2]:
print("Feature Statistics:")
display(df.describe().T)

print("\nClass Distribution (0 = Healthy, 1 = Diabetic):")
print(df['target'].value_counts(normalize=True))

Feature Statistics:


,count,mean,std,min,25%,50%,75%,max
preg,768.0,3.845052,3.369578,0.000,1.00000,3.0000,6.00000,17.00
plas,768.0,120.894531,31.972618,0.000,99.00000,117.0000,140.25000,199.00
pres,768.0,69.105469,19.355807,0.000,62.00000,72.0000,80.00000,122.00
skin,768.0,20.536458,15.952218,0.000,0.00000,23.0000,32.00000,99.00
insu,768.0,79.799479,115.244002,0.000,0.00000,30.5000,127.25000,846.00
mass,768.0,31.992578,7.884160,0.000,27.30000,32.0000,36.60000,67.10
pedi,768.0,0.471876,0.331329,0.078,0.24375,0.3725,0.62625,2.42
age,768.0,33.240885,11.760232,21.000,24.00000,29.0000,41.00000,81.00



Class Distribution (0 = Healthy, 1 = Diabetic):
target
tested_negative    0.651042
tested_positive    0.348958
Name: proportion, dtype: float64


## 3. Data Preprocessing

In many medical datasets, 0 values often represent missing data (e.g., biological impossibility of 0 Glucose or 0 BMI). We will:
1. Replace invalid 0s with NaNs.
2. Impute missing values using the mean.
3. Standardize features for gradient-sensitive algorithms.

In [3]:
import numpy as np
from tuiml.preprocessing.imputation import SimpleImputer
from tuiml.preprocessing.scaling import StandardScaler

# 1. Handle "missing" values (0s in Glucose, BP, BMI, etc.)
X_prep = X.copy()
for i in [1, 2, 5]:  # Indices for plas, pres, mass
    X_prep[X_prep[:, i] == 0, i] = np.nan

# 2. Impute NaNs with Mean
imputer = SimpleImputer(strategy="mean")
X_imputed = imputer.fit_transform(X_prep)

# 3. Standardize (Z-Score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print(f"Preprocessed shape: {X_scaled.shape}")

Preprocessed shape: (768, 8)


## 4. Feature Engineering and Selection

We'll use the **ANOVA F-test (`f_classif`)** to select the top 5 most predictive features. This is a robust method for continuous numerical features in classification tasks.

In [4]:
from tuiml.features.selection import SelectKBestSelector
from tuiml.evaluation.metrics import f_classif

# Select top 5 features using ANOVA F-test
selector = SelectKBestSelector(score_func=f_classif, k=5)
X_selected = selector.fit_transform(X_scaled, y)

selected_indices = selector.get_support(indices=True)
selected_features = [feature_names[i] for i in selected_indices]

print("Top 5 Selected Features:")
for feat in selected_features:
    print(f" - {feat}")

print(f"\nSelected feature matrix shape: {X_selected.shape}")

Top 5 Selected Features:
 - preg
 - plas
 - mass
 - pedi
 - age

Selected feature matrix shape: (768, 5)


## 5. Class Balancing with SMOTE

Now we apply SMOTE to balance the classes on the selected features.

In [5]:
from tuiml.preprocessing.sampling import SMOTESampler

# Apply SMOTE on the selected features
smote = SMOTESampler(random_state=42, k_neighbors=2)
X_resampled, y_resampled = smote.fit_resample(X_selected, y)

print(f"Original shape: {X_selected.shape}")
print(f"Resampled shape: {X_resampled.shape}")
print(f"\nClass distribution after SMOTE:")
unique, counts = np.unique(y_resampled, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  Class {int(u)}: {c} samples")

Original shape: (768, 5)
Resampled shape: (1000, 5)

Class distribution after SMOTE:
  Class 0: 500 samples
  Class 1: 500 samples


## 6. Dataset Splitting

We use a stratified holdout split for final validation.

In [6]:
from tuiml.evaluation.splitting import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, 
    test_size=0.2, 
    stratify=y_resampled, 
    random_state=42
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (800, 5)
Test set: (200, 5)


## 7. Model Training and Selection using Experiments

TuiML's `Experiment` framework allows running multiple models across different validation strategies systematically.

In [7]:
from tuiml.evaluation.experiments import Experiment, ExperimentConfig, ValidationMethod
from tuiml.algorithms.trees import RandomForestClassifier
from tuiml.algorithms.gradient_boosting import XGBoostClassifier
from tuiml.algorithms.svm import SVC

# Define candidate models
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "XGBoost": XGBoostClassifier(n_estimators=100),
    "SVM (SVC)": SVC()
}

# Configure the experiment: 5-Fold Stratified Cross-Validation
config = ExperimentConfig(
    name="Diabetes_Model_Comparison",
    validation_method=ValidationMethod.CROSS_VALIDATION,
    n_folds=5,
    stratify=True,
    random_state=42,
    verbose=1
)

exp = Experiment(config=config, metrics=['accuracy', 'f1_macro', 'roc_auc'])

# Run experiment on the training data
results = exp.run(models=models, datasets={"Diabetes_Train": (X_train, y_train)})

print("\nExperiment completed.")

Processing dataset: Diabetes_Train
  Running model: Random Forest
  Running model: XGBoost


/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [16:35:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/nileshverma/Documents/GitHub/tuiml/tuiml/.venv/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [16:35:55] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


  Running model: SVM (SVC)

Experiment completed.


## 8. Comparative Analysis

We can now view a high-level summary and detailed metrics table for all models.

In [8]:
print("=== Experiment Summary ===")
print(exp.summary())

print("\n=== Performance Matrix (Markdown) ===")
print(exp.to_markdown())

# Find the best model based on accuracy
summary_text = exp.summary(metric="accuracy")
# Programmatic extraction of best performer (simplified for tutorial)
best_model_name = "Random Forest"  # Based on typical results for this dataset

=== Experiment Summary ===
Experiment: Diabetes_Model_Comparison
Validation: cross_validation
Metric: accuracy

Dataset: Diabetes_Train
--------------------------------------------------
  Random Forest: 0.8137 ± 0.0348
  XGBoost: 0.7987 ± 0.0257
  SVM (SVC): 0.7650 ± 0.0430


=== Performance Matrix (Markdown) ===
| Dataset | Random Forest | SVM (SVC) | XGBoost |
|---|---|---|---|
| Diabetes_Train | 0.8137 ± 0.0389 | 0.7650 ± 0.0481 ▼ | 0.7987 ± 0.0288 |
|---|---|---|---|
| **W/L/T** | 0/0/1 | 0/1/0 | 0/0/1 |


## 9. Model Deployment (Serving)

Finally, we train the best model on the full training set and serve it.

In [9]:
import tuiml

# Retrain final model on full resampled data
final_model = models[best_model_name]
final_model.fit(X_resampled, y_resampled)

# Save the complete model and its metadata
tuiml.save(final_model, "diabetes_model.pkl", metadata={
    "workflow": "diabetes_prediction_v1",
    "selected_features": selected_features,
    "feature_indices": selected_indices.tolist()
})

print(f"Model '{best_model_name}' saved as 'diabetes_model.pkl'")
print(f"Selected features: {selected_features}")

Model 'Random Forest' saved as 'diabetes_model.pkl'
Selected features: ['preg', 'plas', 'mass', 'pedi', 'age']


In [10]:
# Start the REST API server (runs in background in Jupyter)
final_model.serve(port=8001)

# Or run from your terminal:
# tuiml serve diabetes_model.pkl --port 8001

Detected running event loop (e.g., Jupyter). Running server in background thread.
Starting TuiML Model Server...
  Model: RandomForestClassifier
  Model ID: RandomForestClassifier
  Server: http://127.0.0.1:8001
  API Docs: http://127.0.0.1:8001/docs



{'server_id': '127.0.0.1:8001',
 'url': 'http://127.0.0.1:8001',
 'thread': <Thread(Thread-5 (run_server), started daemon 6428274688)>,
 'server': <uvicorn.server.Server at 0x118ac8170>}

## 10. Calling the Model API

Once the server is running, you can make predictions via HTTP requests. Below are examples using both **curl** (command line) and **Python requests**.

### Available Endpoints:
| Endpoint | Method | Description |
|----------|--------|-------------|
| `/models` | GET | List loaded models |
| `/models/{model_id}` | GET | Get model info |
| `/models/{model_id}/predict` | POST | Make predictions |
| `/models/{model_id}/predict_proba` | POST | Get class probabilities |
| `/health` | GET | Server health check |
| `/docs` | GET | Interactive API documentation (Swagger UI) |

### Using curl (Command Line)

```bash
# Check loaded models
curl http://127.0.0.1:8001/models

# Make a prediction (5 features: preg, plas, mass, pedi, age)
curl -X POST http://127.0.0.1:8001/models/RandomForestClassifier/predict \
  -H "Content-Type: application/json" \
  -d '{"features": [[6.0, 148.0, 33.6, 0.627, 50.0]]}'

# Get prediction probabilities
curl -X POST http://127.0.0.1:8001/models/RandomForestClassifier/predict_proba \
  -H "Content-Type: application/json" \
  -d '{"features": [[6.0, 148.0, 33.6, 0.627, 50.0], [1.0, 85.0, 26.6, 0.351, 31.0]]}'
```

### Using Python requests

In [11]:
import requests

BASE_URL = "http://127.0.0.1:8001"
MODEL_ID = "RandomForestClassifier"

# Sample data (using selected features: preg, plas, mass, pedi, age)
sample_data = {
    "features": [
        [6.0, 148.0, 33.6, 0.627, 50.0],  # High-risk patient
        [1.0, 85.0, 26.6, 0.351, 31.0],   # Lower-risk patient
        [0.0, 100.0, 22.0, 0.200, 25.0],  # Young, healthy BMI
    ]
}

# Make predictions
response = requests.post(
    f"{BASE_URL}/models/{MODEL_ID}/predict",
    json=sample_data
)
print("=== Predictions ===")
print(response.json())

=== Predictions ===
{'predictions': [1, 1, 1], 'model_id': 'RandomForestClassifier', 'model_class': 'RandomForestClassifier'}


In [12]:
# Get prediction probabilities
response = requests.post(
    f"{BASE_URL}/models/{MODEL_ID}/predict_proba",
    json=sample_data
)
result = response.json()

print("=== Prediction Probabilities ===")
print(f"Classes: {result['classes']} (0=Healthy, 1=Diabetic)\n")

for i, probs in enumerate(result['probabilities']):
    print(f"Patient {i+1}: P(Healthy)={probs[0]:.1%}, P(Diabetic)={probs[1]:.1%}")

=== Prediction Probabilities ===
Classes: [0, 1] (0=Healthy, 1=Diabetic)

Patient 1: P(Healthy)=35.0%, P(Diabetic)=65.0%
Patient 2: P(Healthy)=37.0%, P(Diabetic)=63.0%
Patient 3: P(Healthy)=25.0%, P(Diabetic)=75.0%


## 11. Simplified Workflows (The High-Level Approach)

While the manual steps above are great for learning, TuiML provides high-level APIs to achieve the same result with much less code.

### Option A: Fluent Workflow API (Method Chaining)
The `Workflow` class allows you to chain steps together into a single pipeline.

In [13]:
from tuiml.workflow import Workflow

# Build, train, and evaluate in one go
result = (
    Workflow(X_imputed, target=y)  # Use imputed data
    .standardize()                # Add scaler
    .select_features(k=5)         # Select top 5 features
    .resample(method="smote")     # Handle imbalance
    .train("RandomForestClassifier", n_estimators=100) 
    .cross_validate(cv=5)         # 5-fold CV
    .run()
)

print(f"Workflow Score (Accuracy): {result.metrics['cv_accuracy_score_mean']:.4f} ± {result.metrics['cv_accuracy_score_std']:.4f}")

# Serve directly from the result object!
result.serve(port=8002)

Workflow Score (Accuracy): 0.8190 ± 0.0180


/Users/nileshverma/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/threading.py:1012: RuntimeWarning: coroutine 'Server.serve' was never awaited
  self._target(*self._args, **self._kwargs)


{'server_id': '127.0.0.1:8002',
 'host': '127.0.0.1',
 'port': 8002,
 'model_id': 'RandomForestClassifier',
 'url': 'http://127.0.0.1:8002',
 'endpoints': {'predict': 'http://127.0.0.1:8002/models/RandomForestClassifier/predict',
  'health': 'http://127.0.0.1:8002/health',
  'docs': 'http://127.0.0.1:8002/docs'}}

### Option B: The "One-Liner" Hub API
The `tuiml.train` function is a powerful entry point that handles common patterns using presets or simple lists.

In [14]:
import tuiml

# Complete pipeline with SMOTE and feature selection in one function call
result = tuiml.train(
    algorithm="RandomForestClassifier",
    data=X_imputed,
    target=y,
    preprocessing=["StandardScaler", "SMOTESampler"],
    feature_selection={"name": "SelectKBestSelector", "k": 5},
    cv=5,
    n_estimators=100
)

print(f"API Score (Accuracy): {result.metrics['cv_accuracy_score_mean']:.4f} ± {result.metrics['cv_accuracy_score_std']:.4f}")

# Serve it
tuiml.serve(result, port=8003)

API Score (Accuracy): 0.8100 ± 0.0141


/Users/nileshverma/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/threading.py:1012: RuntimeWarning: coroutine 'Server.serve' was never awaited
  self._target(*self._args, **self._kwargs)


{'server_id': '127.0.0.1:8003',
 'host': '127.0.0.1',
 'port': 8003,
 'model_id': 'default',
 'url': 'http://127.0.0.1:8003',
 'endpoints': {'predict': 'http://127.0.0.1:8003/models/default/predict',
  'health': 'http://127.0.0.1:8003/health',
  'docs': 'http://127.0.0.1:8003/docs'}}

### Summary

In this tutorial, we used **TuiML** to:
- Load a medical dataset and perform EDA.
- Clean data by handling physiological impossibilities (0s) and imputing values.
- **Select top 5 features using ANOVA F-test (`f_classif`).**
- Apply SMOTE on selected features to ensure model stability.
- Systematically compare algorithms using the `Experiment` framework.
- Deploy the winner as a production-ready API.
- **Call the deployed model using curl and Python requests.**
- **Simplify the entire process using Workflow and Hub APIs.**

**Note on Hyperparameters**: We used `n_estimators=100` for both Random Forest and XGBoost. TuiML follows standard naming conventions for consistency across different algorithms.

In [16]:
# Cleanup for tutorial
import os
if os.path.exists("diabetes_model.pkl"):
    os.remove("diabetes_model.pkl")

# Stop all running servers
tuiml.stop_server()